# AIC2026 TEAM-EVAL E0/E1 — Cross-Level Bootstrap

Team-neutral, offline source-pool preparation only. This notebook does not create semantic queries or ground truth.

Required Kaggle inputs:
1. Raw AIC corpus: `/kaggle/input/datasets/nadkli/dataset-aic` (nested roots supported).
2. Offline current repository snapshot containing `src/aic2026_eval`: default `/kaggle/input/datasets/irthn1311/aic2026-team-eval-repository` (nested roots supported).

Optional input: a dataset containing `aic2026_l21_eval_bootstrap.zip`, default search hint `/kaggle/input/datasets/irthn1311/aic2026-l21-eval-bootstrap`. Absence safely produces `SKIPPED_NO_INPUT`.

Internet required: **No**. Output ZIP: `/kaggle/working/aic2026_team_eval_e01_bundle.zip`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from zipfile import ZipFile

DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
REPO_INPUT = Path(os.environ.get('AIC_REPO_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-team-eval-repository'))
L21_INPUT = Path(os.environ.get('AIC_L21_BOOTSTRAP_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-l21-eval-bootstrap'))
OUTPUT_ROOT = Path('/kaggle/working/aic2026_team_eval_e01')
ZIP_PATH = Path('/kaggle/working/aic2026_team_eval_e01_bundle.zip')
print({'data_input': str(DATA_INPUT), 'repo_input': str(REPO_INPUT), 'optional_l21_input': str(L21_INPUT), 'output_zip': str(ZIP_PATH), 'internet_required': False})

In [ ]:
sys.path.insert(0, str(REPO_INPUT / 'src')) if (REPO_INPUT / 'src/aic2026_eval').is_dir() else None
try:
    from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file, resolve_repository_root
except ModuleNotFoundError:
    if not REPO_INPUT.exists(): raise RuntimeError(f'Required offline repository dataset is missing: {REPO_INPUT}')
    markers = list(REPO_INPUT.rglob('src/aic2026_eval/discovery.py'))
    if len(markers) != 1: raise RuntimeError(f'Expected one nested TEAM-EVAL repository marker under {REPO_INPUT}; found {markers}')
    sys.path.insert(0, str(markers[0].parents[2] / 'src'))
    from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file, resolve_repository_root

REPO_ROOT = resolve_repository_root(REPO_INPUT)
DATASET_ROOT = resolve_dataset_root(DATA_INPUT)
L21_ZIP = resolve_named_file(L21_INPUT, 'aic2026_l21_eval_bootstrap.zip', optional=True)
sys.path.insert(0, str(REPO_ROOT / 'src'))
commit_result = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=False)
BUILD_COMMIT = commit_result.stdout.strip() if commit_result.returncode == 0 else os.environ.get('AIC_BUILD_COMMIT', 'UNKNOWN_OFFLINE_SNAPSHOT')
print({'resolved_repo': str(REPO_ROOT), 'resolved_dataset': str(DATASET_ROOT), 'resolved_l21_zip': str(L21_ZIP) if L21_ZIP else None, 'commit': BUILD_COMMIT})

In [ ]:
from aic2026_eval.pipeline import run_bootstrap

RESULT = run_bootstrap(dataset_root=DATASET_ROOT, repository_root=REPO_ROOT, output_root=OUTPUT_ROOT, l21_bootstrap_zip=L21_ZIP, manual_exclude_path=REPO_ROOT / 'configs/eval/manual_exclude_videos.txt', build_commit=BUILD_COMMIT)
print(json.dumps({key: value for key, value in RESULT.items() if key.isupper()}, indent=2))

In [ ]:
CORPUS = RESULT['corpus_summary']
assert CORPUS['status'] == 'PASS' and CORPUS['video_count'] > 0
print('Corpus by source group:', json.dumps(CORPUS['by_source_group'], indent=2))

In [ ]:
USAGE = RESULT['usage_summary']
assert USAGE['status'] == 'PASS'
print('Usage tiers:', json.dumps(USAGE['by_usage_tier'], indent=2), 'scan_mode:', USAGE['repository_scan_mode'])

In [ ]:
print('L21 mapping audit:', json.dumps(RESULT['l21_mapping_summary'], indent=2))
assert RESULT['L21_MAPPING_AUDIT'] in {'PASS', 'PARTIAL', 'SKIPPED'}

In [ ]:
SELECTION, ATLAS = RESULT['selection_report'], RESULT['atlas_index']
assert SELECTION['candidate_count'] == 36
assert SELECTION['blind_candidate_count'] == 24 and SELECTION['sealed_candidate_count'] == 12
assert SELECTION['blind_sealed_video_overlap'] == 0 and SELECTION['t2_t3_used'] is False
assert ATLAS['status'] == 'READY' and ATLAS['atlas_sheet_count'] == 36
print(json.dumps({'selection': SELECTION, 'atlas': ATLAS}, indent=2))

In [ ]:
assert Path(RESULT['zip_path']) == ZIP_PATH and ZIP_PATH.is_file()
with ZipFile(ZIP_PATH) as archive:
    members = archive.namelist()
assert not any(name.endswith(('.mp4', '.npy', '.npz', '.pt', '.pth')) for name in members)
for key in ('TEAM_EVAL_INFRA_STATUS','CORPUS_INVENTORY','CONTAMINATION_CENSUS','L21_MAPPING_AUDIT','HELDOUT_CANDIDATES','BLIND_CANDIDATE_VIDEOS','SEALED_CANDIDATE_VIDEOS','BLIND_SEALED_VIDEO_OVERLAP','ATLAS_STATUS','DENSE_RENDERER_STATUS','PREDICTION_VALIDATOR','SHARED_EVALUATOR','READY_FOR_AI_SEMANTIC_SELECTION'):
    print(f'{key}={RESULT[key]}')
print('BLIND_BENCHMARK_COMPLETE=NO')
print('DOWNLOAD ZIP:', ZIP_PATH, 'size_bytes=', ZIP_PATH.stat().st_size, 'members=', len(members))